# Step 2 - Data Extraction, Cleaning & Analysis (Daft.ie New Homes)

This notebook:
1. Reads a list of property URLs produced by Step 1
2. Visits each URL with Selenium (a real browser), waits for JavaScript to render, then extracts structured data
3. Cleans and standardises the extracted data
4. Adds derived fields (county, province, unique ID)
5. Exports CSVs for further analysis

---
### Why are there 50+ URLs per page in Step 1?
Daft.ie displays 20 property *cards* per search-results page visually,
but each card can represent a **development** containing many individual units
(e.g. a 60-unit apartment block shows as one card but generates one URL per unit type).
The Step 1 scraper collects every individual listing href it finds in the page source,
including links that are rendered off-screen or in hidden pagination prefetch data.
This is correct behaviour - more URLs means more granular data.
Each URL below maps to exactly one addressable listing on daft.ie.
---

In [ ]:
# ===========================================================================
# CELL 1 - IMPORTS AND CONFIGURATION
# ===========================================================================
# All third-party imports are collected here so it is easy to see what
# packages need to be installed (pip install selenium webdriver-manager
# beautifulsoup4 lxml pandas requests).
# ===========================================================================

import re
import json
import time
import random
import hashlib
from datetime import datetime
from pathlib import Path
from typing import Optional

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# ---------------------------------------------------------------------------
# USER CONFIGURATION - edit these paths before running
# ---------------------------------------------------------------------------

# Path to the CSV produced by Step 1 (must contain a column called 'url')
INPUT_CSV = r"C:\xxx\new_homes\daft_listings_newhomes_20260409_214834.csv"

# Directory where all output CSVs will be saved
OUTPUT_DIR = Path(r"C:\Users\danie\OneDrive\Python\Ciaran\Task 2\xxx\new_homes")

# How many listings to scrape in this run.
# Set to None to scrape everything (can take a long time for large lists).
SCRAPE_LIMIT = 10

# If the scraper crashes mid-run, set this to the index it stopped at
# so you can resume without re-scraping from the beginning.
START_AT_INDEX = 0

# Seconds to pause between pages (keeps us polite and avoids bot detection).
MIN_DELAY = 4.5
MAX_DELAY = 6.5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Configuration loaded.")
print(f"Input  : {INPUT_CSV}")
print(f"Output : {OUTPUT_DIR}")
print(f"Limit  : {SCRAPE_LIMIT if SCRAPE_LIMIT else 'ALL'}")

In [ ]:
# ===========================================================================
# CELL 2 - REGEX PATTERNS (defined once, reused throughout)
# ===========================================================================
# Compiling patterns once at module level is faster than recompiling on
# every call inside the extraction function.
# ===========================================================================

# Matches a Euro price like '€ 450,000' or '€450000'
EURO_RE = re.compile(r"\u20ac\s*([\d,]+)")

# Matches an area measurement like '112.28 m²', '112 m2', '112 sqm'
AREA_RE = re.compile(
    r"([\d,.]+)\s*(?:m\u00b2|m2|sqm|sq\.?\s*m)\b",
    re.IGNORECASE
)

# Valid BER ratings as defined by the SEAI (Sustainable Energy Authority of Ireland).
# Anything scraped that is not in this set is treated as a parse error.
VALID_BER = {
    "A1","A2","A3",
    "B1","B2","B3",
    "C1","C2","C3",
    "D1","D2",
    "E1","E2",
    "F","G",
    "EXEMPT"
}

# Maximum plausible view count for a single listing.
# Values above this threshold almost certainly mean a price or year was
# mistakenly captured instead of the view count.
VIEWS_MAX = 500_000

# Plausible size bounds for a single residential unit in Ireland (m2).
AREA_MIN_M2 = 20
AREA_MAX_M2 = 800

print("Regex patterns compiled.")

In [ ]:
# ===========================================================================
# CELL 3 - UNIQUE ID GENERATOR
# ===========================================================================
# Generates a human-readable unique identifier for each listing.
#
# Format: {COUNTY_CODE}{YEAR}{SEQUENCE}
#   e.g. WW2026001  (Wicklow, listed in 2026, 1st record for that county/year)
#        D12026042  (Dublin, listed in 2026, 42nd record)
#        GA2026007  (Galway, 7th record)
#
# Why bother? When data moves into a database or is shared between files,
# having a stable, human-readable key makes joins and lookups much easier
# than relying on row numbers alone. The Daft ID is also stored separately
# as an external reference.
# ===========================================================================

COUNTY_CODES = {
    "Dublin":     "D1",   # D1 distinguishes Dublin from Donegal
    "Cork":       "CK",
    "Galway":     "GA",
    "Limerick":   "LK",
    "Waterford":  "WD",
    "Tipperary":  "TP",
    "Clare":      "CE",
    "Kerry":      "KY",
    "Kildare":    "KE",
    "Meath":      "MH",
    "Louth":      "LH",
    "Wexford":    "WX",
    "Wicklow":    "WW",
    "Kilkenny":   "KK",
    "Laois":      "LS",
    "Offaly":     "OY",
    "Westmeath":  "WH",
    "Carlow":     "CW",
    "Longford":   "LD",
    "Mayo":       "MO",
    "Sligo":      "SO",
    "Roscommon":  "RN",
    "Leitrim":    "LM",
    "Donegal":    "DL",
    "Monaghan":   "MN",
    "Cavan":      "CN",
    "Antrim":     "AM",
}

def generate_unique_id(county_standard: Optional[str], year: int, sequence: int) -> str:
    """
    Build a unique listing identifier.
    county_standard : standardised county name (e.g. 'Wicklow')
    year            : 4-digit integer (e.g. 2026)
    sequence        : 1-based position of this record within its county+year group
    Returns e.g. 'WW2026001'
    """
    code = COUNTY_CODES.get(county_standard, "XX") if county_standard else "XX"
    return f"{code}{year}{sequence:03d}"


def assign_unique_ids(df: pd.DataFrame, year: int) -> pd.Series:
    """
    Vectorised wrapper: assigns an ID to every row based on county and year.
    Rows with the same county get sequential numbers starting from 001.
    """
    ids = []
    county_counters = {}
    for county in df["county_standard"]:
        county_counters[county] = county_counters.get(county, 0) + 1
        ids.append(generate_unique_id(county, year, county_counters[county]))
    return pd.Series(ids, index=df.index)


print("Unique ID generator ready.")
# Quick test
print("Example IDs:")
for county in ["Wicklow", "Dublin", "Galway", "Mayo", None]:
    print(f"  {str(county or 'None'):15} -> {generate_unique_id(county, 2026, 1)}")

In [ ]:
# ===========================================================================
# CELL 4 - FIELD EXTRACTION FUNCTIONS
# ===========================================================================
# extract_fields() is the core function. It receives the full HTML of one
# daft.ie listing page and returns a dictionary of extracted values.
#
# Strategy order for each field:
#   1. Targeted CSS selector (most precise - pinned to a specific element)
#   2. __NEXT_DATA__ JSON blob (Next.js embeds structured data in a script tag)
#   3. Regex over full page text (broadest, most fragile - used as last resort)
#
# When inspecting daft.ie to update selectors:
#   Right-click the element in Chrome > Inspect > right-click the highlighted
#   node in DevTools > Copy > Copy selector  (or Copy JS path).
# ===========================================================================


def _extract_next_data(soup: BeautifulSoup) -> dict:
    """
    Daft.ie is built with Next.js. It embeds all page data as a JSON object
    inside a <script id='__NEXT_DATA__'> tag. This is the most reliable
    source for structured fields because it is machine-generated, not
    formatted for human display.
    Returns the parsed dict, or {} if not found / not parseable.
    """
    tag = soup.select_one("script#__NEXT_DATA__")
    if tag and tag.string:
        try:
            return json.loads(tag.string)
        except Exception:
            pass
    return {}


def _get_listing_dict(next_data: dict) -> dict:
    """
    Navigate to the listing object inside the Next.js data tree.
    The path is: props -> pageProps -> listing
    Returns the listing dict, or {} if not found.
    """
    return (
        next_data
        .get("props", {})
        .get("pageProps", {})
        .get("listing", {})
    ) or {}


def extract_views_and_date_listed(soup: BeautifulSoup) -> tuple:
    """
    Extracts views and date_listed from the 'Ad performance' section.

    The daft.ie HTML for this section looks like:
        <ul class='...'>
          <li class='...'>
            <span ...>Date listed</span>
            <span ...>13/03/2026</span>
          </li>
          <li class='...'>
            <span ...>Views</span>
            <span ...>2,010</span>
          </li>
        </ul>

    Because the CSS class names on daft.ie are auto-generated hashes that
    change with deployments, we locate the <ul> by data-testid='statistics'
    parent and then match by label text, not by class.

    Returns (views: int|None, date_listed: str|None)
    """
    views = None
    date_listed = None

    # Find the 'Ad performance' stats container by its stable data-testid
    stats_section = soup.select_one('[data-testid="statistics"]')
    if stats_section:
        # Each data point is inside an <li>. We read the label span and the
        # value span independently to avoid relying on specific class names.
        for li in stats_section.find_all("li"):
            spans = li.find_all("span")
            if len(spans) >= 2:
                label = spans[0].get_text(strip=True)
                value = spans[1].get_text(strip=True)

                if label == "Views":
                    # Views are formatted like '2,010' - strip commas
                    m = re.search(r"[\d,]+", value)
                    if m:
                        candidate = int(m.group().replace(",", ""))
                        # Reject implausible values (years, prices)
                        if 0 < candidate <= VIEWS_MAX:
                            views = candidate

                elif label == "Date listed":
                    # Format on daft.ie is DD/MM/YYYY - keep as string for now
                    if re.match(r"\d{2}/\d{2}/\d{4}", value):
                        date_listed = value

    # Fallback: scan the full page text for a 'Views' label followed by a number
    if views is None:
        page_text = soup.get_text("\n", strip=True)
        # This pattern looks for 'Views' on one line and a number on the next
        m = re.search(
            r"\bViews\b[^\n]*\n[^\n]*(\d[\d,]*)",
            page_text,
            re.IGNORECASE
        )
        if m:
            candidate = int(m.group(1).replace(",", ""))
            if 0 < candidate <= VIEWS_MAX:
                views = candidate

    return views, date_listed


def extract_ber(soup: BeautifulSoup, page_text: str) -> Optional[str]:
    """
    Extracts BER rating.

    From the pasted HTML we can see the BER SVG has:
        <div aria-hidden='true' aria-label='BER A2' ...>
    Note: aria-hidden='true' means this element is hidden from screen readers
    but the aria-label attribute is still present in the DOM and readable by
    BeautifulSoup. This is the most reliable selector.
    """
    # Strategy 1: aria-label on the BER badge SVG container
    ber_el = soup.find(attrs={"aria-label": re.compile(r"^BER\s", re.IGNORECASE)})
    if ber_el:
        label = ber_el.get("aria-label", "")
        m = re.search(r"BER\s+([A-G]\d?)", label, re.IGNORECASE)
        if m and m.group(1).upper() in VALID_BER:
            return m.group(1).upper()

    # Strategy 2: data-testid="ber" container
    ber_section = soup.select_one('[data-testid="ber"]')
    if ber_section:
        text = ber_section.get_text(" ", strip=True)
        m = re.search(r"\b([A-G]\d?)\b", text.upper())
        if m and m.group(1) in VALID_BER:
            return m.group(1)

    # Strategy 3: look for 'ber_A2_large' style tokens in the full page
    # (these appear in SVG title elements and alt text)
    m = re.search(r"\bber[_\s-]*([A-G]\d?)\b", page_text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_BER:
        return m.group(1).upper()

    return None


def extract_agent_details(soup: BeautifulSoup) -> dict:
    """
    Extracts agent name, agency name, phone number, and PSR licence number.

    From the pasted HTML the contact form section contains:
        data-testid='contact-form-container'
    Inside it there are spans with the agent name, agency, PSR number, and phone.
    We locate these by data-testid attributes and label text rather than
    fragile auto-generated class names.
    """
    agent_name    = None
    agent_agency  = None
    agent_phone   = None
    agent_licence = None

    container = soup.select_one('[data-testid="contact-form-container"]')
    if container:
        page_text = container.get_text("\n", strip=True)

        # Agent name: the first bold span (data-testid='bold-text')
        name_el = container.select_one('[data-testid="bold-text"]')
        if name_el:
            agent_name = name_el.get_text(strip=True)

        # Agency name: the paragraph directly below the agent name
        agency_el = container.select_one('[data-testid="plain-text"]')
        if agency_el:
            agent_agency = agency_el.get_text(strip=True)

        # PSR Licence number appears as text after 'PSR Licence Number:'
        m = re.search(r"PSR Licence Number[:\s]+([\d]+)", page_text, re.IGNORECASE)
        if m:
            agent_licence = m.group(1)

        # Phone number: looks for Irish mobile/landline patterns
        # Irish mobile: 08X XXX XXXX, Irish landline: 0X XXXX XXXX
        m = re.search(r"0[0-9][0-9 ]{7,12}", page_text)
        if m:
            agent_phone = re.sub(r"\s+", " ", m.group()).strip()

    return {
        "agent_name"   : agent_name,
        "agent_agency" : agent_agency,
        "agent_phone"  : agent_phone,
        "agent_licence": agent_licence,
    }


def extract_coordinates(soup: BeautifulSoup, listing_json: dict) -> tuple:
    """
    Extracts latitude and longitude for the property.

    The most reliable source is the __NEXT_DATA__ JSON, which typically
    contains a 'point' or 'coordinates' key inside the listing object.
    Daft.ie also embeds coordinates in a JSON-LD script tag (schema.org format).
    As a last resort we look for a Google Maps embed URL in the HTML.

    Returns (lat: float|None, lng: float|None)
    """
    lat = lng = None

    # Strategy 1: __NEXT_DATA__ listing object
    # Common key paths seen on daft.ie:
    for lat_key, lng_key in [
        ("latitude",  "longitude"),
        ("lat",       "lng"),
        ("lat",       "lon"),
    ]:
        if lat_key in listing_json and lng_key in listing_json:
            try:
                lat = float(listing_json[lat_key])
                lng = float(listing_json[lng_key])
                return lat, lng
            except (TypeError, ValueError):
                pass

    # Check nested 'point' object (GeoJSON style)
    point = listing_json.get("point") or listing_json.get("location") or {}
    if isinstance(point, dict):
        try:
            lat = float(point.get("latitude") or point.get("lat") or "")
            lng = float(point.get("longitude") or point.get("lng") or point.get("lon") or "")
            if lat and lng:
                return lat, lng
        except (TypeError, ValueError):
            pass

    # Strategy 2: JSON-LD schema.org/Place or schema.org/House
    for script in soup.find_all("script", {"type": "application/ld+json"}):
        try:
            data = json.loads(script.string or "")
            geo = data.get("geo") or data.get("location", {}).get("geo", {})
            if geo:
                lat = float(geo.get("latitude", 0))
                lng = float(geo.get("longitude", 0))
                if lat and lng:
                    return lat, lng
        except Exception:
            pass

    # Strategy 3: Google Maps embed URL in the page source
    # e.g. https://maps.google.com/maps?q=53.1234,-6.5678
    m = re.search(
        r"maps[./].*?[?&](?:q|ll|center)=([\-\d.]+)[,+]([\-\d.]+)",
        str(soup)
    )
    if m:
        try:
            lat = float(m.group(1))
            lng = float(m.group(2))
        except ValueError:
            pass

    return lat, lng


def extract_schools(soup: BeautifulSoup) -> tuple:
    """
    Extracts the nearest primary and secondary school names from the
    'Local Area' section.

    From the pasted HTML, daft.ie renders school information in a tab panel
    controlled by buttons with aria-label 'Primary Schools' / 'Secondary Schools'.
    The tab panels are identified by aria-controls='tabpanel-0' and 'tabpanel-1'.
    School names and distances appear in a table inside each panel.

    Note: Selenium must scroll to this section and wait for it to load
    before calling this function, because it is rendered lazily.
    If no data is found, both values will be None.

    Returns (nearest_primary: str|None, nearest_secondary: str|None)
    """
    nearest_primary   = None
    nearest_secondary = None

    # Find tab panels by their id pattern
    primary_panel   = soup.find(id="tabpanel-0")
    secondary_panel = soup.find(id="tabpanel-1")

    for panel, target_var_name in [
        (primary_panel,   "nearest_primary"),
        (secondary_panel, "nearest_secondary")
    ]:
        if panel:
            # School name is typically the first cell of the first data row
            rows = panel.find_all("tr")
            for row in rows:
                cells = row.find_all(["td", "th"])
                if len(cells) >= 1:
                    name = cells[0].get_text(strip=True)
                    if name and name.lower() not in ("school", "name", ""):
                        if target_var_name == "nearest_primary":
                            nearest_primary = name
                        else:
                            nearest_secondary = name
                        break

    return nearest_primary, nearest_secondary


def extract_description(soup: BeautifulSoup) -> Optional[str]:
    """
    Extracts the full property description text.

    From the pasted HTML the description lives inside:
        <section data-testid='description'>
    This is a stable data-testid that should survive class name changes.
    """
    desc_el = soup.select_one('[data-testid="description"]')
    if desc_el:
        return desc_el.get_text(" ", strip=True)
    return None


def extract_daft_id(url: str, listing_json: dict) -> Optional[str]:
    """
    Extracts the Daft listing ID.

    The cleanest source is the URL itself: daft.ie URLs end with the numeric ID.
    e.g. https://www.daft.ie/new-home-for-sale/.../6519420 -> '6519420'
    We also check the __NEXT_DATA__ JSON as a fallback.
    """
    # From URL: last path segment if it is purely numeric
    url_parts = url.rstrip("/").split("/")
    if url_parts and url_parts[-1].isdigit():
        return url_parts[-1]

    # From JSON
    for key in ("id", "listingId", "propertyId"):
        if key in listing_json:
            return str(listing_json[key])

    return None


def extract_area_m2(page_text: str) -> Optional[float]:
    """
    Extracts floor area from page text, validating the result is within
    plausible residential bounds.
    Returns the first match within AREA_MIN_M2 to AREA_MAX_M2.
    """
    for m in AREA_RE.finditer(page_text):
        try:
            val = float(m.group(1).replace(",", ""))
            if AREA_MIN_M2 <= val <= AREA_MAX_M2:
                return val
        except ValueError:
            continue
    return None


def extract_fields(html: str, url: str = "") -> dict:
    """
    Master extraction function. Calls all individual extractors and
    assembles the result into a single flat dictionary.

    Parameters
    ----------
    html : str
        Full rendered HTML from Selenium (driver.page_source).
    url  : str
        The URL of the page (used for Daft ID extraction).

    Returns
    -------
    dict with keys: title, address, daft_id, ber_rating, beds, baths,
                    area_m2, price_eur, price_per_m2_eur, views,
                    date_listed, description, agent_name, agent_agency,
                    agent_phone, agent_licence, lat, lng,
                    nearest_primary_school, nearest_secondary_school
    """
    soup = BeautifulSoup(html, "lxml")

    # Helper: get text content of the first element matching a CSS selector
    def txt(sel):
        el = soup.select_one(sel)
        return el.get_text(" ", strip=True) if el else None

    # Full visible text of the page (used by multiple extractors)
    page_text = soup.get_text("\n", strip=True)

    # --- __NEXT_DATA__ JSON (parsed once, shared across extractors) ---
    next_data    = _extract_next_data(soup)
    listing_json = _get_listing_dict(next_data)

    # --- Title ---
    h1    = soup.find("h1")
    title = h1.get_text(" ", strip=True) if h1 else listing_json.get("title")

    # --- Address ---
    # Try explicit address field first, then subtitle, then derive from title.
    address = (
        txt('[data-testid="address"]') or
        txt('[data-testid="header-subtitle"]') or
        None
    )
    if not address and title and "," in title:
        parts = [p.strip() for p in title.split(",") if p.strip()]
        if len(parts) >= 2:
            address = ", ".join(parts[-2:])

    # --- Daft ID ---
    daft_id = extract_daft_id(url, listing_json)

    # --- Price ---
    price_text = txt('[data-testid="price"]') or txt('[data-testid="header-price"]')
    price_eur  = None
    if price_text:
        m = EURO_RE.search(price_text)
        if m:
            price_eur = int(m.group(1).replace(",", ""))
    if price_eur is None:
        # JSON fallback: price stored as numeric in some page versions
        raw_price = listing_json.get("price")
        if raw_price:
            try:
                price_eur = int(str(raw_price).replace(",", "").replace("\u20ac", "").strip())
            except ValueError:
                pass

    # --- Beds / Baths ---
    beds = baths = None
    m = re.search(r"\b([1-9]|1[0-9]|20)\s*Bed\b", page_text, re.IGNORECASE)
    if m:
        beds = int(m.group(1))
    m = re.search(r"\b([1-9]|1[0-9]|20)\s*Bath\b", page_text, re.IGNORECASE)
    if m:
        baths = int(m.group(1))

    # --- Area ---
    area_m2 = extract_area_m2(page_text)
    if area_m2 is None:
        for key in ("floorArea", "area", "size", "displaySize"):
            v = listing_json.get(key)
            if isinstance(v, (int, float)) and AREA_MIN_M2 <= v <= AREA_MAX_M2:
                area_m2 = float(v)
                break

    # --- Price per m2 ---
    price_per_m2_eur = None
    m = re.search(r"Price\s*per\s*m\u00b2\s*[:\-]?\s*\u20ac\s*([\d,]+)", page_text, re.IGNORECASE)
    if m:
        price_per_m2_eur = int(m.group(1).replace(",", ""))
    elif price_eur is not None and area_m2:
        price_per_m2_eur = round(price_eur / area_m2)

    # --- BER ---
    ber_rating = extract_ber(soup, page_text)

    # --- Views & Date Listed ---
    views, date_listed = extract_views_and_date_listed(soup)

    # --- Description ---
    description = extract_description(soup)

    # --- Agent Details ---
    agent = extract_agent_details(soup)

    # --- Coordinates ---
    lat, lng = extract_coordinates(soup, listing_json)

    # --- Schools ---
    nearest_primary, nearest_secondary = extract_schools(soup)

    return {
        "title"                   : title,
        "address"                 : address,
        "daft_id"                 : daft_id,
        "ber_rating"              : ber_rating,
        "beds"                    : beds,
        "baths"                   : baths,
        "area_m2"                 : area_m2,
        "price_eur"               : price_eur,
        "price_per_m2_eur"        : price_per_m2_eur,
        "views"                   : views,
        "date_listed"             : date_listed,
        "description"             : description,
        "agent_name"              : agent["agent_name"],
        "agent_agency"            : agent["agent_agency"],
        "agent_phone"             : agent["agent_phone"],
        "agent_licence"           : agent["agent_licence"],
        "lat"                     : lat,
        "lng"                     : lng,
        "nearest_primary_school"  : nearest_primary,
        "nearest_secondary_school": nearest_secondary,
    }


print("All extraction functions loaded.")

In [ ]:
# ===========================================================================
# CELL 5 - LOAD URLs
# ===========================================================================
# Read the URL list produced by Step 1 and apply the scrape limit.
# ===========================================================================

df_urls = pd.read_csv(INPUT_CSV)

# Ensure the URL column exists
if "url" not in df_urls.columns:
    raise ValueError(f"Expected a 'url' column in {INPUT_CSV}. Found: {df_urls.columns.tolist()}")

all_urls = df_urls["url"].dropna().tolist()

# Apply optional start index (for resuming after a crash)
urls_to_scrape = all_urls[START_AT_INDEX:]

# Apply optional scrape limit
if SCRAPE_LIMIT is not None:
    urls_to_scrape = urls_to_scrape[:SCRAPE_LIMIT]

print(f"Total URLs in file : {len(all_urls)}")
print(f"Starting at index  : {START_AT_INDEX}")
print(f"URLs to scrape     : {len(urls_to_scrape)}")
print()
print("First 5 URLs:")
for u in urls_to_scrape[:5]:
    print(f"  {u}")

In [ ]:
# ===========================================================================
# CELL 6 - SELENIUM SETUP
# ===========================================================================
# We use a real Chrome browser (not headless by default) because daft.ie
# loads views and stats via JavaScript after the initial page render.
# A headless browser sometimes triggers anti-bot checks, so the visible
# window is left open during scraping.
#
# user-data-dir: reuses a saved Chrome profile so login cookies and
# preferences persist between runs. This helps avoid repeated CAPTCHA
# challenges. The path below is a throw-away profile - it will be created
# on first run.
# ===========================================================================

options = Options()
options.add_argument("--window-size=1400,900")
options.add_argument("--user-data-dir=/tmp/selenium-daft-profile")

# Removes the 'Chrome is being controlled by automated software' banner,
# which some sites detect as a bot signal.
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

# Uncomment the line below to run headlessly (no visible browser window).
# Only do this once the scraper is stable - headless mode can miss lazy-
# loaded sections like schools and ad performance statistics.
# options.add_argument("--headless")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# Hide the webdriver property from JavaScript so it is less detectable
driver.execute_cdp_cmd(
    "Page.addScriptToEvaluateOnNewDocument",
    {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"}
)

print("Chrome browser started.")

In [ ]:
# ===========================================================================
# CELL 7 - MAIN SCRAPE LOOP
# ===========================================================================
# Visits each URL, waits for the page to fully render, then calls
# extract_fields() on the rendered HTML.
#
# The random sleep between pages (MIN_DELAY to MAX_DELAY seconds) serves two
# purposes:
#   1. Gives JavaScript time to render dynamic sections (views, schools)
#   2. Mimics human browsing speed and reduces bot-detection risk
#
# If a Cloudflare challenge page is detected the scraper pauses an extra 7
# seconds and tries to read the page again after the challenge completes
# (Cloudflare typically auto-solves in a real browser within 5 seconds).
# ===========================================================================

rows = []
total = len(urls_to_scrape)

for i, url in enumerate(urls_to_scrape, start=1):
    print(f"[{i:4d}/{total}] {url}")

    try:
        driver.get(url)
        # Wait for main page render
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

        html = driver.page_source

        # Detect Cloudflare bot challenge
        if "_cf_chl_opt" in html or "Checking your browser" in html:
            print("         Cloudflare detected - waiting 8 seconds...")
            time.sleep(8)
            html = driver.page_source

        # Scroll to the bottom to trigger lazy-loaded sections
        # (schools, coordinates map, ad performance stats)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)  # allow lazy sections to load after scroll
        html = driver.page_source  # re-capture after scroll

        data = extract_fields(html, url=url)

    except Exception as e:
        print(f"         ERROR: {e}")
        data = {}

    data["url"]          = url
    data["date_scraped"] = datetime.now().strftime("%Y-%m-%d")
    rows.append(data)

    # Print a one-line summary for this listing
    price_str = f"EUR {data.get('price_eur'):,}" if data.get('price_eur') else "price NA"
    views_str = f"views={data.get('views')}" if data.get('views') else "views=NA"
    print(f"         {data.get('title','(no title)')[:55]}  |  {price_str}  |  {views_str}")

    # Brief additional pause between pages
    time.sleep(random.uniform(0.2, 0.5))

driver.quit()
print(f"\nScraping complete. {len(rows)} records collected.")

In [ ]:
# ===========================================================================
# CELL 8 - BUILD DATAFRAME AND ENFORCE TYPES
# ===========================================================================
# Convert the list of dicts into a pandas DataFrame and enforce column types.
#
# Numeric columns are cast to nullable integer (Int64) rather than float.
# This preserves the distinction between 0 and missing - a property with
# 0 views is different from one where views could not be scraped (NA).
# ===========================================================================

COLUMN_ORDER = [
    "url",
    "daft_id",
    "title",
    "address",
    "date_listed",
    "date_scraped",
    "views",
    "price_eur",
    "beds",
    "baths",
    "area_m2",
    "price_per_m2_eur",
    "ber_rating",
    "agent_name",
    "agent_agency",
    "agent_phone",
    "agent_licence",
    "lat",
    "lng",
    "nearest_primary_school",
    "nearest_secondary_school",
    "description",
]

df = pd.DataFrame(rows)

# Add any missing columns as NA so the DataFrame always has the same shape
for col in COLUMN_ORDER:
    if col not in df.columns:
        df[col] = pd.NA

df = df[COLUMN_ORDER]  # enforce column order

# Cast types
df["date_scraped"] = pd.to_datetime(df["date_scraped"], errors="coerce")
df["date_listed"]  = pd.to_datetime(df["date_listed"], format="%d/%m/%Y", errors="coerce")

for int_col in ["views", "price_eur", "beds", "baths", "price_per_m2_eur"]:
    df[int_col] = pd.to_numeric(df[int_col], errors="coerce").astype("Int64")

for float_col in ["area_m2", "lat", "lng"]:
    df[float_col] = pd.to_numeric(df[float_col], errors="coerce")

print(f"DataFrame shape: {df.shape}")
print("\nColumn summary (non-null counts):")
print(df.notna().sum().to_string())
print("\nFirst record:")
print(df.iloc[0].to_string())

In [ ]:
# ===========================================================================
# CELL 9 - COUNTY AND PROVINCE STANDARDISATION
# ===========================================================================
# Irish property addresses end in 'Co. Cork', 'Co. Wicklow', 'Dublin 4' etc.
# We extract the last comma-separated segment and map it to a canonical
# county name and province.
# ===========================================================================

def standardise_county(x) -> Optional[str]:
    """
    Converts raw county text like 'Co. Wicklow' or 'County Dublin' into
    a clean canonical name like 'Wicklow' or 'Dublin'.
    """
    if pd.isna(x):
        return None

    s = re.sub(r"\s+", " ", str(x).strip())

    # Remove suffixes like 'City', 'Town', 'City Centre' from the end
    s = re.sub(r"\s+(City Centre|City Center|City|Town)\s*$", "", s, flags=re.IGNORECASE).strip()

    # Collapse all Dublin variants (Dublin 1, Dublin 4, D4, DUB1 etc.) to 'Dublin'
    if re.match(r"^(DUBLIN\s*\d*|DUB\s*\d+)$", s.upper().replace(" ", "")):
        return "Dublin"

    # Strip 'Co.' or 'County' prefix
    s = re.sub(r"^(Co\.?|County)\s+", "", s, flags=re.IGNORECASE).strip()

    COUNTY_NAMES = {
        "CORK":"Cork","KILDARE":"Kildare","WEXFORD":"Wexford","WICKLOW":"Wicklow",
        "WESTMEATH":"Westmeath","MEATH":"Meath","GALWAY":"Galway","LOUTH":"Louth",
        "WATERFORD":"Waterford","LIMERICK":"Limerick","KILKENNY":"Kilkenny",
        "LAOIS":"Laois","OFFALY":"Offaly","SLIGO":"Sligo","CLARE":"Clare",
        "TIPPERARY":"Tipperary","KERRY":"Kerry","CAVAN":"Cavan","MAYO":"Mayo",
        "LEITRIM":"Leitrim","ANTRIM":"Antrim","DONEGAL":"Donegal",
        "MONAGHAN":"Monaghan","ROSCOMMON":"Roscommon","CARLOW":"Carlow",
        "LONGFORD":"Longford","DUBLIN":"Dublin",
    }
    return COUNTY_NAMES.get(s.upper(), s.title())


PROVINCE_MAP = {
    # Leinster (12 counties)
    "Dublin":"Leinster","Kildare":"Leinster","Wicklow":"Leinster",
    "Wexford":"Leinster","Kilkenny":"Leinster","Laois":"Leinster",
    "Offaly":"Leinster","Meath":"Leinster","Westmeath":"Leinster",
    "Carlow":"Leinster","Louth":"Leinster","Longford":"Leinster",
    # Munster (6 counties)
    "Cork":"Munster","Waterford":"Munster","Limerick":"Munster",
    "Kerry":"Munster","Tipperary":"Munster","Clare":"Munster",
    # Connacht (5 counties)
    "Galway":"Connacht","Mayo":"Connacht","Sligo":"Connacht",
    "Roscommon":"Connacht","Leitrim":"Connacht",
    # Ulster (9 counties, 3 in ROI)
    "Donegal":"Ulster","Monaghan":"Ulster","Cavan":"Ulster",
    "Antrim":"Ulster",
}

# Extract county from the last segment of the address
df["county"] = (
    df["address"].fillna("").astype(str)
    .str.split(",").str[-1].str.strip()
)

df["county_standard"] = df["county"].apply(standardise_county)
df["province"]        = df["county_standard"].map(PROVINCE_MAP).fillna("Unknown")

print("County/province standardisation complete.")
print(df[["address","county_standard","province"]].head(10).to_string())

In [ ]:
# ===========================================================================
# CELL 10 - UNIQUE ID ASSIGNMENT
# ===========================================================================
# Assigns the generated unique ID (e.g. WW2026001) to each row.
# The ID encodes: county abbreviation + year of scrape + sequence number.
#
# This is our internal key. The daft_id column holds the external reference
# (the numeric ID from the daft.ie URL). Having both is useful:
#   - daft_id links back to the live listing on daft.ie
#   - unique_id is stable if/when the listing is removed from daft.ie
# ===========================================================================

scrape_year = datetime.now().year

df["unique_id"] = assign_unique_ids(df, scrape_year)

# Move unique_id to the front
cols = ["unique_id"] + [c for c in df.columns if c != "unique_id"]
df = df[cols]

print("Unique IDs assigned.")
print(df[["unique_id","daft_id","county_standard","title"]].head(10).to_string())

In [ ]:
# ===========================================================================
# CELL 11 - DATA QUALITY SENSE-CHECK
# ===========================================================================
# Prints a report of rows with suspicious values.
# These rows are NOT removed - they are flagged for manual review.
# The flag column is also exported to a separate CSV.
# ===========================================================================

print("DATA QUALITY REPORT")
print("=" * 60)

flags = {}

# Views: 2025/2026 is the current year leaking in; >500k is almost certainly a price
bad_views_year = df[df["views"].isin([2024, 2025, 2026])]
bad_views_high = df[df["views"] > 500_000]
if len(bad_views_year):
    print(f"VIEWS - {len(bad_views_year)} row(s) where views equals a recent year (scrape error):")
    print(bad_views_year[["unique_id","title","views"]].to_string()); print()
    flags.update({i: "views_is_year" for i in bad_views_year.index})
if len(bad_views_high):
    print(f"VIEWS - {len(bad_views_high)} row(s) with views > 500,000 (likely a price was matched):")
    print(bad_views_high[["unique_id","title","views"]].to_string()); print()
    flags.update({i: "views_too_high" for i in bad_views_high.index})

# Area: average Irish new home is ~100 m2; flag below 20 or above 300
bad_area_low  = df[df["area_m2"].notna() & (df["area_m2"] < 20)]
bad_area_high = df[df["area_m2"].notna() & (df["area_m2"] > 300)]
if len(bad_area_low):
    print(f"AREA - {len(bad_area_low)} row(s) with area_m2 < 20 (implausibly small):")
    print(bad_area_low[["unique_id","title","area_m2"]].to_string()); print()
    flags.update({i: "area_too_small" for i in bad_area_low.index})
if len(bad_area_high):
    print(f"AREA - {len(bad_area_high)} row(s) with area_m2 > 300 (unusually large - verify):")
    print(bad_area_high[["unique_id","title","area_m2"]].to_string()); print()
    flags.update({i: "area_too_large" for i in bad_area_high.index})

# Price per m2: EUR 39/m2 is nonsensical (seen in previous output)
bad_ppm2 = df[df["price_per_m2_eur"].notna() & (df["price_per_m2_eur"] < 500)]
if len(bad_ppm2):
    print(f"PRICE/M2 - {len(bad_ppm2)} row(s) with price_per_m2 < EUR 500 (probably a parse error):")
    print(bad_ppm2[["unique_id","title","price_eur","area_m2","price_per_m2_eur"]].to_string()); print()
    flags.update({i: "price_per_m2_too_low" for i in bad_ppm2.index})

# BER: flag values not in the SEAI-defined set
bad_ber = df[df["ber_rating"].notna() & ~df["ber_rating"].isin(VALID_BER)]
if len(bad_ber):
    print(f"BER - {len(bad_ber)} row(s) with unrecognised BER values:")
    print(bad_ber[["unique_id","title","ber_rating"]].to_string()); print()
    flags.update({i: "ber_invalid" for i in bad_ber.index})

# Add flag column to main df (empty string = no flag)
df["data_quality_flag"] = df.index.map(flags).fillna("")

if not flags:
    print("No quality issues found.")

print("\nSummary statistics:")
print(df[["views","price_eur","beds","baths","area_m2","price_per_m2_eur"]].describe().round(1).to_string())

In [ ]:
# ===========================================================================
# CELL 12 - SORT AND FINALISE
# ===========================================================================

df = df.sort_values(
    by=["province", "county_standard", "address"],
    ascending=[True, True, True],
    na_position="last"
).reset_index(drop=True)

print(f"Final DataFrame: {len(df)} rows x {len(df.columns)} columns")
print("\nProvince breakdown:")
print(
    df.groupby("province", dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .to_string(index=False)
)

In [ ]:
# ===========================================================================
# CELL 13 - EXPORT ALL CSVs
# ===========================================================================
# Five CSVs are exported:
#
#  1. daft_listings_full_{ts}.csv     - every column, every row (master file)
#  2. daft_views_date_{ts}.csv        - date_listed + views only (as requested)
#  3. daft_top20_views_{ts}.csv       - top 20 listings by view count
#  4. daft_by_province_{ts}.csv       - aggregated stats by province
#  5. daft_quality_flags_{ts}.csv     - rows with data quality issues
# ===========================================================================

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Full master file
f1 = OUTPUT_DIR / f"daft_listings_full_{ts}.csv"
df.to_csv(f1, index=False)
print(f"Full data         : {f1}")

# 2. Views + date listed (two clean columns as requested, plus identifiers)
f2 = OUTPUT_DIR / f"daft_views_date_{ts}.csv"
df[["unique_id","daft_id","date_listed","views","title","url"]].to_csv(f2, index=False)
print(f"Views + date      : {f2}")
print("\nViews + date preview:")
print(df[["unique_id","date_listed","views"]].head(10).to_string())

# 3. Top 20 by views
f3 = OUTPUT_DIR / f"daft_top20_views_{ts}.csv"
(
    df.nlargest(20, "views")[
        ["unique_id","daft_id","title","address","date_listed","views",
         "price_eur","beds","baths","area_m2","ber_rating",
         "county_standard","province","url"]
    ].to_csv(f3, index=False)
)
print(f"\nTop 20 by views   : {f3}")

# 4. Province-level aggregated stats
f4 = OUTPUT_DIR / f"daft_by_province_{ts}.csv"
(
    df.groupby(["province","county_standard"])
    .agg(
        listings       =("unique_id","count"),
        avg_price      =("price_eur","mean"),
        median_price   =("price_eur","median"),
        avg_price_pm2  =("price_per_m2_eur","mean"),
        avg_views      =("views","mean"),
        median_views   =("views","median"),
    )
    .round(0)
    .reset_index()
    .sort_values("listings", ascending=False)
    .to_csv(f4, index=False)
)
print(f"Province summary  : {f4}")

# 5. Quality flags (only if any issues were found)
flagged = df[df["data_quality_flag"] != ""]
if len(flagged):
    f5 = OUTPUT_DIR / f"daft_quality_flags_{ts}.csv"
    flagged[["unique_id","daft_id","title","url","views","area_m2",
              "price_eur","price_per_m2_eur","ber_rating","data_quality_flag"]]\
        .to_csv(f5, index=False)
    print(f"Quality flags     : {f5}  ({len(flagged)} rows need review)")
else:
    print("No quality flags - skipping flags CSV.")

print("\nAll exports complete.")

In [ ]:
# ===========================================================================
# CELL 14 - ANALYSIS: PRICE PER M2 BY PROVINCE AND COUNTY
# ===========================================================================

print("PRICE PER M2 BY PROVINCE")
print("=" * 50)
province_summary = (
    df.groupby("province")["price_per_m2_eur"]
    .agg(
        min_price_per_m2   ="min",
        max_price_per_m2   ="max",
        median_price_per_m2="median",
        avg_price_per_m2   ="mean",
        count              ="count",
    )
    .round(0)
    .reset_index()
)
print(province_summary.to_string(index=False))

print("\n\nPRICE PER M2 BY COUNTY")
print("=" * 50)
county_summary = (
    df.groupby("county_standard")["price_per_m2_eur"]
    .agg(
        min_price_per_m2   ="min",
        max_price_per_m2   ="max",
        median_price_per_m2="median",
        avg_price_per_m2   ="mean",
        count              ="count",
    )
    .round(0)
    .reset_index()
    .sort_values("avg_price_per_m2", ascending=False)
)
print(county_summary.to_string(index=False))

In [ ]:
# ===========================================================================
# CELL 15 - ANALYSIS: AD PERFORMANCE (VIEWS) DASHBOARD
# ===========================================================================

print("DAFT.IE AD PERFORMANCE DASHBOARD")
print("=" * 60)
print(f"Analysis date          : {datetime.now().strftime('%d %B %Y')}")
print(f"Properties analysed    : {len(df)}")
print()

v = df["views"].dropna()
if len(v) > 0:
    print("VIEWS OVERVIEW")
    print(f"  Properties with views  : {len(v)}")
    print(f"  Total views captured   : {v.sum():,.0f}")
    print(f"  Average views per ad   : {v.mean():,.0f}")
    print(f"  Median views per ad    : {v.median():,.0f}")
    print(f"  Top performer          : {v.max():,.0f} views")
    print(f"  Top 10% threshold      : {v.quantile(0.9):,.0f} views")
    print()

    print("TOP 5 PROPERTIES BY VIEWS")
    top5 = df.nlargest(5, "views")[["unique_id","title","views","date_listed","province"]]
    print(top5.to_string(index=False))
    print()

    print("AVERAGE VIEWS BY PROVINCE")
    prov_views = (
        df.groupby("province")["views"]
        .agg(["mean","median","count"])
        .round(0)
        .sort_values("mean", ascending=False)
    )
    print(prov_views.to_string())
else:
    print("No views data available. Check the extract_views_and_date_listed function.")